# Step 2 — 01. Preprocessing and model smoke test

등록된 factory로 실제 PyTorch 모델을 만들고 고정된 정렬 crop에서
raw 512D, raw norm, L2-normalized embedding 및 Grad-CAM target
layer를 검증합니다. checkpoint를 학습하거나 변경하지 않습니다.

In [7]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [8]:
import json
import numpy as np

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    read_model_spec,
)

MODEL_SPEC_PATH = "data/interim/lfw/summary.json"       # 00에서 생성한 JSON
SMOKE_CROPS_NPZ = None       # aligned_faces: uint8 [N,112,112,3]
DEVICE = "cpu"               # GPU 확인 후 "cuda"로 변경 가능
MAX_SMOKE_IMAGES = 8

In [9]:
if EXECUTE_STAGE:
    if MODEL_SPEC_PATH is None or SMOKE_CROPS_NPZ is None:
        raise RuntimeError("MODEL_SPEC_PATH와 SMOKE_CROPS_NPZ를 지정하세요.")
    spec = read_model_spec(MODEL_SPEC_PATH, verify_checkpoint=True)
    if spec.family != MODEL_NAME:
        raise ValueError("MODEL_NAME과 ModelSpec family가 다릅니다.")
    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)
    bundle = np.load(SMOKE_CROPS_NPZ, allow_pickle=False)
    aligned_faces = bundle["aligned_faces"][:MAX_SMOKE_IMAGES]
    output = adapter.embed(aligned_faces)
    _ = adapter.target_layer

    unit_norms = np.linalg.norm(output.normalized_embedding, axis=1)
    smoke_summary = {
        "model_uid": spec.model_uid,
        "checkpoint_sha256": spec.checkpoint.sha256,
        "preprocess_hash": spec.preprocessing.preprocess_hash,
        "sample_count": int(len(aligned_faces)),
        "raw_shape": list(output.raw_embedding.shape),
        "raw_norm_min": float(output.raw_norm.min()),
        "raw_norm_max": float(output.raw_norm.max()),
        "maximum_unit_norm_error": float(np.max(np.abs(unit_norms - 1.0))),
        "target_layer": spec.target_layer,
        "status": "validated",
    }
    if WRITE_OUTPUTS:
        destination = (
            PROJECT_ROOT
            / "runs/step2/model_validation"
            / spec.model_uid
            / "smoke_summary.json"
        )
        if destination.exists():
            raise FileExistsError(f"기존 결과를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(
            json.dumps(smoke_summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
else:
    smoke_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
smoke_summary

{'status': 'not_executed', 'reason': 'EXECUTE_STAGE=False'}

이 smoke test가 통과해도 세 loss의 인과 비교가 성립하는 것은
아닙니다. 이후 정량 실험은 모델별 새 embedding/PCA/PQ lineage에서
수행해야 하며 Step 1 ONNX artifact에 합치지 않습니다.